# Mini-Projeto Avaliativo - Módulo 2

### Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import time

from sklearn.ensemble import RandomForestClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.datasets import fetch_openml

## Fase 1: Carregamento e Análise Exploratória de Imagens (EDA)

#### 1.1 Importação e carregamento do MNIST

In [ ]:
print("Baixando o dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
print("Finalizou o processo de baixar o dataset MNIST!")


#### 1.2 Separação entre features e target

In [ ]:
X = mnist.data
y = mnist.target.astype(int)

print("Features (X):", X.shape)
print("Target (y):", y.shape)

### 1.3 Dimensionalidade dos dados

In [ ]:
print("Dimensionalidade dos dados:")
print(f"X: {X.shape}")
print(f"y: {y.shape}")

print("\nQuantidade de imagens:", X.shape[0])
print("Quantidade de features por imagem:", X.shape[1])

#### 1.4 Distribuição das classes

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()

print("Distribuição das classes:")
print(class_counts)

In [ ]:
plt.figure(figsize=(8, 5))

class_counts.plot(kind='bar')

plt.title('Distribuição das Classes no MNIST')
plt.xlabel('Dígito')
plt.ylabel('Quantidade de imagens')
plt.xticks(rotation=0)

plt.show()

#### 1.5 Visualização dos dígitos

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for digit, ax in enumerate(axes.ravel()):
    # Localiza a primeira imagem correspondente ao dígito
    index = np.where(y == digit)[0][0]
    
    # Reconstrói o vetor de 784 pixels para 28x28
    image = X[index].reshape(28, 28)
    
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Dígito: {digit}')
    ax.axis('off')

plt.tight_layout()
plt.show()

#### 1.6 Estrutura dos pixels

In [ ]:
print("Valor mínimo dos pixels:", X.min())
print("Valor máximo dos pixels:", X.max())

In [ ]:
sample_index = 0

image = X[sample_index].reshape(28, 28)

print("Dimensão da imagem:", image.shape)

print("\nMatriz de pixels:")
print(image)

In [ ]:
plt.figure(figsize=(5, 5))

plt.imshow(image, cmap='gray')

plt.title(f'Dígito: {y[sample_index]}')
plt.axis('off')

plt.show()

#### 1.7 Interpretação dos dados

O dataset MNIST é composto por imagens de dígitos manuscritos com resolução de **28 × 28 pixels**. Cada pixel possui uma intensidade que varia de **0 a 255**, onde valores próximos de 0 representam regiões mais escuras e valores próximos de 255 representam regiões mais claras.

Embora a imagem possua originalmente duas dimensões (28 × 28), os modelos de Machine Learning utilizados neste projeto recebem os dados em formato vetorial. Dessa forma, cada imagem é transformada em um vetor de **784 features (28 × 28 = 784)**.

A matriz `X` contém as imagens vetorizadas, enquanto o vetor `y` contém o rótulo correspondente a cada imagem, representando um dos dez dígitos possíveis, de **0 a 9**.

A análise da distribuição das classes nos permite verificar se existe algum desequilíbrio relevante entre os diferentes dígitos antes da etapa de treinamento dos modelos. 

Após análise, conseguimos ver que não há desequilíbrio relevante entre os diferentes dígitos.


## Fase 2: Pipeline de Pré-processamento e Divisão dos Dados

#### 2.1 Divisão estratificada dos dados

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    random_state=42,
    stratify=y_temp
)

### 2.2 Verificação das dimensões

In [ ]:
print("Dimensões dos conjuntos:")

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"\nX_val: {X_val.shape}")
print(f"y_val: {y_val.shape}")

print(f"\nX_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

### 2.3 Verificação da estratificação

In [ ]:
print("Distribuição percentual das classes:\n")

print("Treino:")
print(
    pd.Series(y_train)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nValidação:")
print(
    pd.Series(y_val)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nTeste:")
print(
    pd.Series(y_test)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

### 2.4 Normalização dos pixels

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

### 2.5 Verificação da normalização

In [ ]:
print("Após a normalização:")

print(f"Valor mínimo em X_train: {X_train.min()}")
print(f"Valor máximo em X_train: {X_train.max()}")

print(f"\nValor mínimo em X_val: {X_val.min()}")
print(f"Valor máximo em X_val: {X_val.max()}")

print(f"\nValor mínimo em X_test: {X_test.min()}")
print(f"Valor máximo em X_test: {X_test.max()}")

### 2.6 Visualização de imagem após normalização

In [ ]:
sample_index = 0

plt.figure(figsize=(5, 5))

plt.imshow(
    X_train[sample_index].reshape(28, 28),
    cmap='gray'
)

plt.title(f'Dígito: {y_train[sample_index]}')
plt.axis('off')

plt.show()

### 2.7 Análise

Após o pré-processamento, o dataset foi dividido de forma estratificada em treino, validação e teste, mantendo a proporção das dez classes entre os conjuntos.

Os pixels foram normalizados de uma escala original de 0–255 para 0.0–1.0, preparando os dados para o treinamento dos modelos. A normalização coloca todas as características em uma escala comum, facilitando o processo de otimização e contribuindo para a convergência de modelos sensíveis à escala dos dados.

Essa transformação também é especialmente importante para algoritmos baseados em distância, como o KNN, pois evita que diferenças de escala prejudiquem o cálculo das distâncias entre as observações.

O conjunto de teste permanece separado para a avaliação final dos modelos, evitando que suas informações sejam utilizadas durante o processo de desenvolvimento e ajuste.

## Fase 3: Implementação e Treinamento dos 3 Modelos

### 3.1 KNN

#### Treinamento e avaliação das configurações

In [ ]:

knn_configs = [
    {"n_neighbors": 3, "weights": "uniform"},
    {"n_neighbors": 5, "weights": "uniform"},
    {"n_neighbors": 7, "weights": "distance"},
]

knn_results = []

for config in knn_configs:
    model = KNeighborsClassifier(
        n_neighbors=config["n_neighbors"],
        weights=config["weights"]
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    knn_results.append({
        "n_neighbors": config["n_neighbors"],
        "weights": config["weights"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

knn_results_df = pd.DataFrame(knn_results)
knn_results_df

#### Escolha do melhor KNN

In [ ]:
best_knn_config = knn_results_df.loc[
    knn_results_df["val_accuracy"].idxmax()
]

best_knn_config

#### Treinamento do modelo final

In [ ]:
best_knn = KNeighborsClassifier(
    n_neighbors=int(best_knn_config["n_neighbors"]),
    weights=best_knn_config["weights"]
)

start_time = time.time()
best_knn.fit(X_train, y_train)
knn_training_time = time.time() - start_time

print(f"Tempo de treinamento: {knn_training_time:.2f} segundos")

### 3.2 Random Forest

#### Treinamento e avaliação das configurações

In [ ]:
rf_configs = [
    {"n_estimators": 100, "max_depth": 15},
    {"n_estimators": 150, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
]

rf_results = []

for config in rf_configs:
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42,
        n_jobs=-1
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    rf_results.append({
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

rf_results_df = pd.DataFrame(rf_results)
rf_results_df

#### Escolha da melhor configuração

In [ ]:
best_rf_config = rf_results_df.loc[
    rf_results_df["val_accuracy"].idxmax()
]

best_rf_config

#### Treinamento do modelo final

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=int(best_rf_config["n_estimators"]),
    max_depth=(
        int(best_rf_config["max_depth"])
        if pd.notna(best_rf_config["max_depth"])
        else None
    ),
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
best_rf.fit(X_train, y_train)
rf_training_time = time.time() - start_time

print(f"Tempo de treinamento: {rf_training_time:.2f} segundos")

### 3.3 MLP

#### Treinamento e avaliação das configurações

In [ ]:
mlp_configs = [
    {"hidden_layer_sizes": (64,), "alpha": 0.0001},
    {"hidden_layer_sizes": (128,), "alpha": 0.0001},
    {"hidden_layer_sizes": (128,), "alpha": 0.001},
]

mlp_results = []

for config in mlp_configs:
    model = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        alpha=config["alpha"],
        max_iter=50,
        random_state=42,
        early_stopping=True
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    mlp_results.append({
        "hidden_layer_sizes": config["hidden_layer_sizes"],
        "alpha": config["alpha"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

mlp_results_df = pd.DataFrame(mlp_results)
mlp_results_df

#### Escolha da melhor configuração

In [ ]:
best_mlp_config = mlp_results_df.loc[
    mlp_results_df["val_accuracy"].idxmax()
]

best_mlp_config

#### Treinamento do modelo final

In [ ]:
best_mlp = MLPClassifier(
    hidden_layer_sizes=best_mlp_config["hidden_layer_sizes"],
    alpha=best_mlp_config["alpha"],
    max_iter=20,
    random_state=42,
    early_stopping=True
)

start_time = time.time()
best_mlp.fit(X_train, y_train)
mlp_training_time = time.time() - start_time

print(f"Tempo de treinamento: {mlp_training_time:.2f} segundos")

### 3.4 Comparação dos modelos

#### Comparação de desempenho e tempo de treinamento

In [ ]:
model_comparison = pd.DataFrame({
    "Modelo": ["KNN", "Random Forest", "MLP"],
    "Accuracy Validação": [
        best_knn_config["val_accuracy"],
        best_rf_config["val_accuracy"],
        best_mlp_config["val_accuracy"]
    ],
    "Tempo Treinamento (s)": [
        knn_training_time,
        rf_training_time,
        mlp_training_time
    ]
})

model_comparison.sort_values(
    "Accuracy Validação",
    ascending=False
)

### 3.5 Análise de overfitting

#### Comparação entre treino e validação

In [ ]:
overfitting_comparison = pd.DataFrame({
    "Modelo": ["KNN", "Random Forest", "MLP"],
    "Accuracy Treino": [
        best_knn_config["train_accuracy"],
        best_rf_config["train_accuracy"],
        best_mlp_config["train_accuracy"]
    ],
    "Accuracy Validação": [
        best_knn_config["val_accuracy"],
        best_rf_config["val_accuracy"],
        best_mlp_config["val_accuracy"]
    ]
})

overfitting_comparison["Diferença"] = (
    overfitting_comparison["Accuracy Treino"]
    - overfitting_comparison["Accuracy Validação"]
)

overfitting_comparison

#### Análise

Após o treinamento e ajuste dos três modelos, o **MLP apresentou o melhor resultado no conjunto de validação, com 97,69% de acurácia**, seguido pelo KNN com 97,11% e pelo Random Forest com 96,64%.

O KNN apresentou a menor diferença entre as acurácias de treinamento e validação, indicando boa capacidade de generalização e baixo indício de overfitting. O MLP apresentou uma diferença moderada de aproximadamente 2,13 pontos percentuais, mantendo, porém, a melhor acurácia de validação. Já o Random Forest apresentou a maior diferença, de 3,31 pontos percentuais, indicando maior tendência ao overfitting.

Considerando a acurácia de validação como principal critério de seleção, o **MLP foi escolhido como o modelo de melhor desempenho nesta etapa**. Entretanto, os três modelos serão avaliados posteriormente no conjunto de teste independente para verificar seu desempenho final e capacidade de generalização.


## Fase 4: Avaliação Comparativa de Desempenho

## Fase 5: Teste dos Modelos em Cenários Atípicos

### Fase 5.1 - Desafio (A) Treinamento Restrito com Classes Ocultadas (Class Masking)

### Fase 5.2 - Desafio (B) Teste de Generalização Extrema (Inferência OOD)

### Fase 5.3 - Desafio (C) Inferência com Imagens Manuscritas Próprias